# 🚀 Feature Engineering 완전 완성 강의

## ML Mentoring - Feature Engineering A to Z

---

### 📋 목차

| 단계 | 내용 | 난이도 |
|------|------|--------|
| 1️⃣ 기초 | 정규화, 표준화, 로버스트 스케일링, 범주형 인코딩 | ⭐ |
| 2️⃣ 중급 | 결측치 처리, Feature Selection, 다항식 특성 | ⭐⭐ |
| 3️⃣ 고급 | PCA, Feature Importance | ⭐⭐⭐ |
| 4️⃣ 실습 | 통합 Pipeline | ⭐⭐⭐⭐ |

---

## 🔧 환경 설정 및 데이터 준비

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import (
    MinMaxScaler, StandardScaler, RobustScaler,
    LabelEncoder, OneHotEncoder, PolynomialFeatures
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, f_classif, f_regression
)
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

matplotlib.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.grid'] = True

print('✅ 모든 라이브러리 로드 완료!')
print(f'NumPy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')

In [ ]:
# ============================================================
# 실습용 데이터 생성
# numeric_cols[25:35] 개념: 35개 피처 중 25~34번 인덱스 10개 사용
# ============================================================
np.random.seed(42)
n_samples = 500

# 교육 목적: 실제 데이터셋처럼 많은 컬럼 중 일부를 선택하는 상황을 재현합니다.
# numeric_cols[25:35]는 '컬럼 목록의 25번~34번 인덱스' 슬라이싱을 의미합니다.
# (실무에서 수십~수백 개 컬럼 중 특정 범위의 컬럼을 선택하는 패턴)
all_numeric_data = np.random.randn(n_samples, 35)

# 실제 세계 특성 반영: 스케일 차이, 이상치, 결측치
all_numeric_data[:, 0] *= 1000   # 큰 스케일
all_numeric_data[:, 1] *= 0.001  # 작은 스케일
all_numeric_data[:, 2] = np.abs(all_numeric_data[:, 2]) * 10  # 양수
all_numeric_data[:, 3] += 50    # 평균 이동

# 이상치 추가
outlier_idx = np.random.choice(n_samples, 20, replace=False)
all_numeric_data[outlier_idx, 0] *= 10

all_cols = [f'feature_{i:02d}' for i in range(35)]
df_full = pd.DataFrame(all_numeric_data, columns=all_cols)

# numeric_cols[25:35] 사용
numeric_cols = all_cols  # 전체 컬럼 리스트
selected_cols = numeric_cols[25:35]  # 25~34번 인덱스
df = df_full[selected_cols].copy()

# 결측치 추가 (실습용)
for col in df.columns[:3]:
    mask = np.random.random(n_samples) < 0.1
    df.loc[mask, col] = np.nan

# 범주형 변수 추가
df['category'] = np.random.choice(['A', 'B', 'C', 'D'], n_samples)
df['size'] = np.random.choice(['Small', 'Medium', 'Large'], n_samples)

# 타겟 변수 생성 (분류용)
y = (df[selected_cols[0]].fillna(0) + df[selected_cols[1]].fillna(0) > 0).astype(int)

print('=' * 60)
print('📊 데이터 준비 완료!')
print('=' * 60)
print(f'선택된 컬럼 (numeric_cols[25:35]): {selected_cols}')
print(f'\n데이터 크기: {df.shape}')
print(f'수치형 특성: {len(selected_cols)}개')
print(f'범주형 특성: 2개 (category, size)')
print(f'타겟 변수 분포:')
print(y.value_counts())
print('\n처음 5행:')
df.head()

---
# 1️⃣ 기초 단계 (Basic)

## Feature Engineering이란?

**Feature Engineering**은 원시 데이터(raw data)를 머신러닝 모델이 더 잘 학습할 수 있는 형태로 변환하는 과정입니다.

```
원시 데이터 → Feature Engineering → 모델 입력
  [복잡, 비정형]     [변환/생성/선택]    [정형, 최적화]
```

### 왜 Feature Engineering이 중요한가?

| 문제 | 해결책 | 효과 |
|------|--------|------|
| 스케일 차이 (0.001 ~ 1000) | 정규화/표준화 | 학습 안정성 ↑ |
| 이상치 (Outlier) | 로버스트 스케일링 | 이상치 영향 ↓ |
| 범주형 데이터 ('A','B','C') | 인코딩 | 수치 변환 |
| 결측치 (NaN) | 대치/제거 | 데이터 완성도 ↑ |
| 고차원 데이터 | 차원 축소 | 계산 효율 ↑ |

In [ ]:
# 현재 데이터의 문제점 확인
print('📊 수치형 특성 기본 통계:')
print(df[selected_cols].describe().round(3))
print('\n❗ 스케일 차이가 큰 것을 확인할 수 있습니다!')
print(f'\n결측치 개수:')
print(df[selected_cols].isnull().sum())

# 스케일 차이 시각화
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for i, col in enumerate(selected_cols):
    ax = axes[i // 5][i % 5]
    df[col].dropna().hist(bins=30, ax=ax, color='steelblue', alpha=0.7)
    ax.set_title(f'{col}\n(mean={df[col].mean():.2f})', fontsize=9)
    ax.tick_params(labelsize=8)
plt.suptitle('원본 데이터 분포 (Feature별)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 1.1 정규화 (Min-Max Normalization)

### 📖 이론

**정규화**는 모든 특성을 **[0, 1]** 범위로 변환하는 방법입니다.

$$x_{normalized} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

### 특징

| 장점 | 단점 |
|------|------|
| 모든 값이 [0, 1] 범위로 제한 | 이상치에 민감 |
| 직관적인 해석 가능 | 새 데이터 범위 벗어날 수 있음 |
| 신경망, KNN, SVM에 적합 | 분포 형태 유지 안 됨 |

### 언제 사용?
- 특성의 범위가 알려져 있을 때
- 신경망 (Neural Network) 학습 시
- 이미지 픽셀값 처리 (0~255 → 0~1)

In [ ]:
# ============================================================
# Min-Max Normalization 실습
# ============================================================
df_num = df[selected_cols].copy()

# 결측치 임시 처리 (평균 대치)
df_num_filled = df_num.fillna(df_num.mean())

# MinMaxScaler 적용
scaler_minmax = MinMaxScaler()
df_normalized = pd.DataFrame(
    scaler_minmax.fit_transform(df_num_filled),
    columns=selected_cols
)

# 수동 계산으로 검증 (첫 번째 특성)
col = selected_cols[0]
x_min = df_num_filled[col].min()
x_max = df_num_filled[col].max()
manual_norm = (df_num_filled[col].iloc[0] - x_min) / (x_max - x_min)

print('=' * 60)
print(f'📐 수동 계산 검증 ({col}):')
print(f'  원본값:         {df_num_filled[col].iloc[0]:.4f}')
print(f'  x_min:          {x_min:.4f}')
print(f'  x_max:          {x_max:.4f}')
print(f'  수동 계산:      {manual_norm:.4f}')
print(f'  sklearn 결과:   {df_normalized[col].iloc[0]:.4f}')
print(f'  일치 여부:      {abs(manual_norm - df_normalized[col].iloc[0]) < 1e-10}')
print('=' * 60)
print('\n정규화 후 통계:')
print(df_normalized.describe().round(4))

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for col in selected_cols[:3]:
    axes[0].hist(df_num_filled[col], bins=30, alpha=0.5, label=col, density=True)
    axes[1].hist(df_normalized[col], bins=30, alpha=0.5, label=col, density=True)

axes[0].set_title('정규화 전 (원본)', fontsize=12)
axes[1].set_title('정규화 후 [0, 1]', fontsize=12)
for ax in axes:
    ax.legend(fontsize=8)
    ax.set_xlabel('값')
    ax.set_ylabel('밀도')
plt.tight_layout()
plt.show()

---
## 1.2 표준화 (Standardization / Z-score Normalization)

### 📖 이론

**표준화**는 데이터를 **평균 0, 표준편차 1**인 표준 정규분포로 변환합니다.

$$z = \frac{x - \mu}{\sigma}$$

- $\mu$ (mu): 특성의 평균 (mean)
- $\sigma$ (sigma): 특성의 표준편차 (standard deviation)

### 특징

| 장점 | 단점 |
|------|------|
| 분포의 형태 유지 | 범위 제한 없음 |
| 이상치 영향 적음 (정규화보다) | 직관적 해석 어려울 수 있음 |
| 선형 모델, PCA에 필수 | 정규분포 가정 |

### 언제 사용?
- 선형 회귀, 로지스틱 회귀
- PCA (주성분 분석)
- SVM (서포트 벡터 머신)
- 특성들의 분포가 정규분포에 가까울 때

In [ ]:
# ============================================================
# Standardization (Z-score) 실습
# ============================================================
scaler_std = StandardScaler()
df_standardized = pd.DataFrame(
    scaler_std.fit_transform(df_num_filled),
    columns=selected_cols
)

# 수동 계산으로 검증
col = selected_cols[0]
mu = df_num_filled[col].mean()
sigma = df_num_filled[col].std()
manual_std = (df_num_filled[col].iloc[0] - mu) / sigma

print('=' * 60)
print(f'📐 수동 계산 검증 ({col}):')
print(f'  원본값:         {df_num_filled[col].iloc[0]:.4f}')
print(f'  평균 (mu):      {mu:.4f}')
print(f'  표준편차:       {sigma:.4f}')
print(f'  z-score:        {manual_std:.4f}')
print(f'  sklearn 결과:   {df_standardized[col].iloc[0]:.4f}')
print('=' * 60)
print('\n표준화 후 통계 (평균≈0, 표준편차≈1):')
print(df_standardized.describe().round(4))

# 정규화 vs 표준화 비교
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
col = selected_cols[0]

axes[0].hist(df_num_filled[col], bins=30, color='gray', alpha=0.7, density=True)
axes[0].set_title(f'원본\n(mean={df_num_filled[col].mean():.2f}, std={df_num_filled[col].std():.2f})', fontsize=10)

axes[1].hist(df_normalized[col], bins=30, color='steelblue', alpha=0.7, density=True)
axes[1].set_title('정규화 (Min-Max)\nmean≈0.5, range=[0,1]', fontsize=10)

axes[2].hist(df_standardized[col], bins=30, color='tomato', alpha=0.7, density=True)
axes[2].set_title('표준화 (Z-score)\nmean=0, std=1', fontsize=10)

for ax in axes:
    ax.set_xlabel('값')
    ax.set_ylabel('밀도')
plt.suptitle('스케일링 방법 비교', fontsize=14)
plt.tight_layout()
plt.show()

---
## 1.3 로버스트 스케일링 (Robust Scaling)

### 📖 이론

**로버스트 스케일링**은 **중앙값(median)과 IQR**을 사용하여 이상치에 강건한 스케일링을 수행합니다.

$$x_{robust} = \frac{x - Q_2}{Q_3 - Q_1}$$

- $Q_1$: 1사분위수 (25th percentile)
- $Q_2$: 중앙값 (50th percentile, median)
- $Q_3$: 3사분위수 (75th percentile)
- $IQR = Q_3 - Q_1$: 사분위 범위

### 핵심 원리: IQR (Interquartile Range)

```
  |------ IQR ------|
  Q1               Q3
  25%     Q2      75%
          (median)
  ← 이상치 │ 정상 범위 │ 이상치 →
```

### 특징

| 장점 | 단점 |
|------|------|
| 이상치 영향 최소화 | 범위 제한 없음 |
| 중앙값 기반 → 왜도 데이터에 강건 | 이상치 제거 안 됨 |

### 언제 사용?
- 데이터에 이상치가 많을 때
- 수입, 가격 등 왜도(skewed) 분포 데이터

In [ ]:
# ============================================================
# Robust Scaling 실습 (이상치 데이터로 비교)
# ============================================================

# 이상치가 있는 데이터 생성
np.random.seed(42)
data_with_outliers = np.concatenate([
    np.random.normal(0, 1, 490),
    np.array([100, -100, 200, -200, 150, -150, 300, -300, 500, -500])
])

# 각 스케일러 적용
data_2d = data_with_outliers.reshape(-1, 1)
scaled_minmax = MinMaxScaler().fit_transform(data_2d).flatten()
scaled_std = StandardScaler().fit_transform(data_2d).flatten()
scaled_robust = RobustScaler().fit_transform(data_2d).flatten()

# 수동 계산 검증
q1 = np.percentile(data_with_outliers, 25)
q2 = np.percentile(data_with_outliers, 50)
q3 = np.percentile(data_with_outliers, 75)
iqr = q3 - q1
manual_robust_0 = (data_with_outliers[0] - q2) / iqr

print('=' * 60)
print('📐 로버스트 스케일링 수동 계산:')
print(f'  Q1 (25%):   {q1:.4f}')
print(f'  Q2 (중앙값): {q2:.4f}')
print(f'  Q3 (75%):   {q3:.4f}')
print(f'  IQR:        {iqr:.4f}')
print(f'  첫 번째 값 (원본): {data_with_outliers[0]:.4f}')
print(f'  수동 계산:         {manual_robust_0:.4f}')
print(f'  sklearn 결과:      {scaled_robust[0]:.4f}')
print('=' * 60)

# 이상치 영향 비교 시각화
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(data_with_outliers, bins=50, color='gray', alpha=0.7)
axes[0, 0].set_title('원본 데이터 (이상치 포함)', fontsize=11)

axes[0, 1].hist(scaled_minmax, bins=50, color='steelblue', alpha=0.7)
axes[0, 1].set_title('Min-Max 정규화\n(이상치로 인해 정상값이 압축됨)', fontsize=11)

axes[1, 0].hist(scaled_std, bins=50, color='tomato', alpha=0.7)
axes[1, 0].set_title('표준화 (Z-score)\n(이상치 영향 약간 있음)', fontsize=11)

axes[1, 1].hist(scaled_robust, bins=50, color='green', alpha=0.7)
axes[1, 1].set_title('로버스트 스케일링\n(이상치에 강건)', fontsize=11)

for ax in axes.flatten():
    ax.set_xlabel('값')
    ax.set_ylabel('빈도')
plt.suptitle('이상치 존재 시 스케일링 방법 비교', fontsize=13)
plt.tight_layout()
plt.show()

---
## 1.4 범주형 인코딩 (Categorical Encoding)

### 📖 이론

머신러닝 모델은 숫자만 이해합니다. 범주형 변수를 숫자로 변환해야 합니다.

#### Label Encoding
각 범주에 정수를 할당합니다.
$$A → 0, \quad B → 1, \quad C → 2, \quad D → 3$$

#### One-Hot Encoding
각 범주를 이진 벡터로 변환합니다.
```
A → [1, 0, 0, 0]
B → [0, 1, 0, 0]
C → [0, 0, 1, 0]
D → [0, 0, 0, 1]
```

### 비교

| 방법 | 장점 | 단점 | 사용 시기 |
|------|------|------|-----------|
| Label Encoding | 간단, 메모리 효율 | 순서 관계 암묵적 생성 | 트리 모델, 순서 있는 범주 |
| One-Hot Encoding | 순서 관계 없음 | 고차원 (카디널리티↑→문제) | 선형 모델, 범주 수가 적을 때 |

In [ ]:
# ============================================================
# Label Encoding 실습
# ============================================================
le = LabelEncoder()
category_label = le.fit_transform(df['category'])

print('=' * 60)
print('🏷️  Label Encoding 결과:')
print(f'원본 클래스: {le.classes_}')
print(f'인코딩 매핑: {dict(zip(le.classes_, le.transform(le.classes_)))}')
print(f'\n원본 샘플 (처음 10개): {df["category"].values[:10].tolist()}')
print(f'인코딩 결과:           {category_label[:10].tolist()}')
print('=' * 60)

# One-Hot Encoding
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
category_ohe = ohe.fit_transform(df[['category']])
ohe_cols = [f'category_{c}' for c in ohe.categories_[0]]
df_ohe = pd.DataFrame(category_ohe, columns=ohe_cols, dtype=int)

print('\n🎯 One-Hot Encoding 결과 (처음 5행):')
sample = pd.concat([df[['category']].reset_index(drop=True), df_ohe], axis=1)
print(sample.head(10).to_string())

# 시각화: 인코딩 방법 비교
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 원본 분포
cat_counts = df['category'].value_counts()
axes[0].bar(cat_counts.index, cat_counts.values, color='steelblue', alpha=0.8)
axes[0].set_title('원본 범주형 변수', fontsize=11)
axes[0].set_xlabel('Category')
axes[0].set_ylabel('빈도')

# Label Encoding 분포
pd.Series(category_label).value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='tomato', alpha=0.8
)
axes[1].set_title('Label Encoding 결과', fontsize=11)
axes[1].set_xlabel('인코딩 값')
axes[1].set_ylabel('빈도')

# One-Hot Encoding 히트맵
sns.heatmap(
    df_ohe.head(20).T,
    ax=axes[2],
    cmap='Blues',
    cbar=False,
    linewidths=0.5,
    yticklabels=ohe_cols
)
axes[2].set_title('One-Hot Encoding (처음 20행)', fontsize=11)
axes[2].set_xlabel('샘플 인덱스')

plt.suptitle('범주형 인코딩 비교', fontsize=13)
plt.tight_layout()
plt.show()

---
# 2️⃣ 중급 단계 (Intermediate)

---
## 2.1 결측치 처리 (Missing Value Handling)

### 📖 이론

결측치(Missing Value, NaN)는 실제 데이터에서 매우 흔하게 발생합니다.

#### 결측치의 유형
| 유형 | 설명 | 예시 |
|------|------|------|
| **MCAR** (완전 무작위 결측) | 무작위로 발생 | 설문 실수 |
| **MAR** (무작위 결측) | 다른 변수와 관련 | 나이에 따른 수입 누락 |
| **MNAR** (비무작위 결측) | 결측값 자체와 관련 | 극단적 값 미보고 |

#### 처리 방법
1. **제거**: 결측치가 있는 행/열 삭제
2. **단순 대치**: 평균, 중앙값, 최빈값으로 대치
3. **KNN 대치**: 유사한 이웃의 값으로 대치
4. **모델 기반**: 예측 모델로 결측치 예측

In [ ]:
# ============================================================
# 결측치 처리 실습
# ============================================================
df_missing = df[selected_cols].copy()

print('=' * 60)
print('📊 결측치 현황:')
missing_info = pd.DataFrame({
    '결측치 수': df_missing.isnull().sum(),
    '결측치 비율 (%)': (df_missing.isnull().sum() / len(df_missing) * 100).round(2)
})
print(missing_info[missing_info['결측치 수'] > 0])
print('=' * 60)

# 방법 1: 제거
df_drop = df_missing.dropna()
print(f'\n방법 1 - 제거 후 크기: {df_drop.shape} (원본: {df_missing.shape})')
print(f'손실된 데이터: {len(df_missing) - len(df_drop)}행 ({(len(df_missing) - len(df_drop))/len(df_missing)*100:.1f}%)')

# 방법 2: 평균 대치
imputer_mean = SimpleImputer(strategy='mean')
df_mean_imputed = pd.DataFrame(
    imputer_mean.fit_transform(df_missing),
    columns=selected_cols
)

# 방법 3: 중앙값 대치
imputer_median = SimpleImputer(strategy='median')
df_median_imputed = pd.DataFrame(
    imputer_median.fit_transform(df_missing),
    columns=selected_cols
)

# 방법 4: KNN 대치
imputer_knn = KNNImputer(n_neighbors=5)
df_knn_imputed = pd.DataFrame(
    imputer_knn.fit_transform(df_missing),
    columns=selected_cols
)

print('\n✅ 각 대치 방법 적용 완료!')
print('\n각 방법별 특성 (feature_25) 통계 비교:')
col = selected_cols[0]
comparison = pd.DataFrame({
    '원본 (결측포함)': df_missing[col].describe(),
    '평균 대치': df_mean_imputed[col].describe(),
    '중앙값 대치': df_median_imputed[col].describe(),
    'KNN 대치': df_knn_imputed[col].describe()
})
print(comparison.round(4))

In [ ]:
# 결측치 처리 시각화
col = selected_cols[0]
mask_missing = df_missing[col].isnull()

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# 원본 (결측치 위치 표시)
axes[0].hist(df_missing[col].dropna(), bins=30, color='gray', alpha=0.7, label='실제값', density=True)
axes[0].set_title(f'원본 ({mask_missing.sum()}개 결측)', fontsize=10)
axes[0].legend()

# 평균 대치
axes[1].hist(df_mean_imputed[col][~mask_missing], bins=30, color='steelblue', alpha=0.5, label='원본값', density=True)
axes[1].hist(df_mean_imputed[col][mask_missing], bins=10, color='orange', alpha=0.8, label='대치값(평균)', density=True)
axes[1].set_title('평균 대치\n(대치값이 하나의 점에 집중)', fontsize=10)
axes[1].legend(fontsize=8)

# 중앙값 대치
axes[2].hist(df_median_imputed[col][~mask_missing], bins=30, color='steelblue', alpha=0.5, label='원본값', density=True)
axes[2].hist(df_median_imputed[col][mask_missing], bins=10, color='green', alpha=0.8, label='대치값(중앙값)', density=True)
axes[2].set_title('중앙값 대치', fontsize=10)
axes[2].legend(fontsize=8)

# KNN 대치
axes[3].hist(df_knn_imputed[col][~mask_missing], bins=30, color='steelblue', alpha=0.5, label='원본값', density=True)
axes[3].hist(df_knn_imputed[col][mask_missing], bins=10, color='tomato', alpha=0.8, label='대치값(KNN)', density=True)
axes[3].set_title('KNN 대치\n(분포 더 자연스러움)', fontsize=10)
axes[3].legend(fontsize=8)

plt.suptitle(f'{col} 결측치 처리 방법 비교', fontsize=13)
plt.tight_layout()
plt.show()

# 결측치 히트맵
fig, ax = plt.subplots(figsize=(12, 4))
missing_matrix = df_missing[selected_cols].isnull().astype(int)
sns.heatmap(missing_matrix.T, ax=ax, cmap='Reds', cbar_kws={'label': '결측치 (1=결측)'},
            xticklabels=False, yticklabels=selected_cols)
ax.set_title('결측치 패턴 시각화 (빨간색 = 결측)', fontsize=12)
ax.set_xlabel('샘플 인덱스')
plt.tight_layout()
plt.show()

---
## 2.2 Feature Selection (특성 선택)

### 📖 이론

**Feature Selection**은 모델에 필요한 중요한 특성만 선택하는 과정입니다.

#### 필요성
- **차원의 저주 (Curse of Dimensionality)**: 특성이 많을수록 모델이 복잡해짐
- **과적합 방지**: 불필요한 특성 제거
- **계산 효율**: 학습 속도 향상
- **해석 가능성**: 중요 특성 파악

#### 방법

| 방법 | 기준 | 예시 |
|------|------|------|
| **분산 기반** | 분산이 낮은 특성 제거 | VarianceThreshold |
| **상관관계 기반** | 타겟과 상관관계 높은 특성 선택 | Pearson, Spearman |
| **통계 검정 기반** | 통계적 유의성 기반 | SelectKBest, chi2 |
| **모델 기반** | 모델 중요도 활용 | Feature Importance |
| **RFE** | 재귀적 특성 제거 | RFE, RFECV |

In [ ]:
# ============================================================
# Feature Selection 실습
# ============================================================
X = df_knn_imputed[selected_cols].values  # 결측치 처리된 데이터 사용
y_target = y.values

print('=' * 60)
print('🔍 Feature Selection 실습')
print('=' * 60)

# 방법 1: 분산 기반 선택 (VarianceThreshold)
var_threshold = VarianceThreshold(threshold=0.1)
X_var = var_threshold.fit_transform(X)
selected_var = [selected_cols[i] for i in range(len(selected_cols)) if var_threshold.get_support()[i]]

print(f'\n[1] 분산 기반 선택 (threshold=0.1):')
print(f'  선택된 특성 수: {X_var.shape[1]}/{len(selected_cols)}')
var_df = pd.DataFrame({'특성': selected_cols, '분산': var_threshold.variances_,
                        '선택': var_threshold.get_support()})
print(var_df.sort_values('분산', ascending=False).to_string())

# 방법 2: SelectKBest (f_classif)
k = 7
selector_kbest = SelectKBest(score_func=f_classif, k=k)
X_kbest = selector_kbest.fit_transform(X, y_target)
selected_kbest = [selected_cols[i] for i in range(len(selected_cols)) if selector_kbest.get_support()[i]]

print(f'\n[2] SelectKBest (k={k}, ANOVA F-test):')
print(f'  선택된 특성: {selected_kbest}')
scores_df = pd.DataFrame({'특성': selected_cols, 'F-score': selector_kbest.scores_,
                           'p-value': selector_kbest.pvalues_,
                           '선택': selector_kbest.get_support()})
print(scores_df.sort_values('F-score', ascending=False).round(4).to_string())

In [ ]:
# 상관관계 분석
df_corr = df_knn_imputed[selected_cols].copy()
df_corr['target'] = y_target

# 타겟과의 상관관계
target_corr = df_corr.corr()['target'].drop('target').abs().sort_values(ascending=False)

print('\n[3] 상관관계 기반 선택 (|r| > 0.1):')
corr_threshold = 0.1
selected_corr = target_corr[target_corr > corr_threshold].index.tolist()
print(f'  선택된 특성: {selected_corr}')
print(f'\n  타겟과의 상관계수:')
print(target_corr.round(4))

# 시각화: Feature Selection 결과 비교
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 분산 막대 그래프
var_sorted = var_df.sort_values('분산', ascending=True)
colors_var = ['tomato' if not s else 'steelblue' for s in var_sorted['선택']]
axes[0].barh(var_sorted['특성'], var_sorted['분산'], color=colors_var, alpha=0.8)
axes[0].axvline(x=0.1, color='red', linestyle='--', label='threshold=0.1')
axes[0].set_title('분산 기반 선택\n(파란색=선택, 빨간색=제거)', fontsize=10)
axes[0].legend(fontsize=8)

# F-score 막대 그래프
scores_sorted = scores_df.sort_values('F-score', ascending=True)
colors_f = ['tomato' if not s else 'steelblue' for s in scores_sorted['선택']]
axes[1].barh(scores_sorted['특성'], scores_sorted['F-score'], color=colors_f, alpha=0.8)
axes[1].set_title(f'SelectKBest (k={k})\n(파란색=선택)', fontsize=10)

# 상관계수 히트맵
corr_matrix = df_corr.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, ax=axes[2], mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', vmin=-1, vmax=1, square=True, linewidths=0.5,
            annot_kws={'size': 7})
axes[2].set_title('상관관계 히트맵', fontsize=10)

plt.suptitle('Feature Selection 방법 비교', fontsize=13)
plt.tight_layout()
plt.show()

---
## 2.3 다항식 특성 생성 (Polynomial Feature Generation)

### 📖 이론

**다항식 특성**은 기존 특성의 조합으로 새로운 특성을 생성합니다.

#### degree=2인 경우 ($x_1, x_2$ 두 개의 특성)

$$[x_1, x_2] \rightarrow [1, x_1, x_2, x_1^2, x_1 x_2, x_2^2]$$

#### 일반식
$$n \text{개 특성, degree=}d \Rightarrow \binom{n+d}{d} \text{개 특성 생성}$$

| 특성 수 | degree | 생성되는 특성 수 |
|---------|--------|------------------|
| 2 | 2 | 6 |
| 5 | 2 | 21 |
| 10 | 2 | 66 |
| 10 | 3 | 286 |

### 주의사항
- degree가 높을수록 특성이 기하급수적으로 증가
- 과적합(Overfitting) 위험 증가
- 선형 모델의 비선형 관계 학습 가능

In [ ]:
# ============================================================
# Polynomial Features 실습
# ============================================================
from math import comb

# 처음 3개 특성만 사용 (시연용)
X_small = df_knn_imputed[selected_cols[:3]].values

print('=' * 60)
print('🔢 다항식 특성 생성')
print('=' * 60)

for degree in [2, 3]:
    poly = PolynomialFeatures(degree=degree, include_bias=True)
    X_poly = poly.fit_transform(X_small)
    expected = comb(3 + degree, degree)
    print(f'\ndegree={degree}:')
    print(f'  원본 특성 수:   3')
    print(f'  생성된 특성 수: {X_poly.shape[1]} (이론값: {expected})')
    print(f'  특성 이름: {poly.get_feature_names_out(["f1","f2","f3"]).tolist()}')

# degree=2 상세 실습
poly2 = PolynomialFeatures(degree=2, include_bias=False)
X_poly2 = poly2.fit_transform(X_small)
feature_names = poly2.get_feature_names_out(selected_cols[:3]).tolist()

df_poly = pd.DataFrame(X_poly2[:5], columns=feature_names)
print('\n다항식 특성 샘플 (처음 5행):')
print(df_poly.round(4).to_string())

# 비선형 관계 시각화
np.random.seed(42)
x_demo = np.linspace(-3, 3, 100)
y_demo = 2 * x_demo**2 - 3 * x_demo + 1 + np.random.normal(0, 0.5, 100)

from sklearn.linear_model import LinearRegression
x_2d = x_demo.reshape(-1, 1)

# 선형 피팅
lr = LinearRegression().fit(x_2d, y_demo)
y_linear = lr.predict(x_2d)

# 다항식 피팅
poly_demo = PolynomialFeatures(degree=2, include_bias=True)
x_poly_demo = poly_demo.fit_transform(x_2d)
lr_poly = LinearRegression().fit(x_poly_demo, y_demo)
y_poly_pred = lr_poly.predict(x_poly_demo)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(x_demo, y_demo, alpha=0.5, label='데이터', s=20)
axes[0].plot(x_demo, y_linear, 'r-', linewidth=2, label='선형 회귀')
axes[0].set_title('선형 회귀 (부적합)', fontsize=11)
axes[0].legend()

axes[1].scatter(x_demo, y_demo, alpha=0.5, label='데이터', s=20)
axes[1].plot(x_demo, y_poly_pred, 'g-', linewidth=2, label='다항식 회귀 (degree=2)')
axes[1].set_title('다항식 회귀 (적합)', fontsize=11)
axes[1].legend()

plt.suptitle('다항식 특성으로 비선형 관계 학습', fontsize=13)
plt.tight_layout()
plt.show()

---
# 3️⃣ 고급 단계 (Advanced)

---
## 3.1 PCA (주성분 분석, Principal Component Analysis)

### 📖 이론

**PCA**는 데이터의 분산을 최대한 보존하면서 차원을 축소하는 기법입니다.

#### 핵심 아이디어
데이터의 **분산이 가장 큰 방향**을 새로운 축(주성분, Principal Component)으로 선택합니다.

#### 수학적 원리
1. 데이터 중심화: $X_{centered} = X - \bar{X}$
2. 공분산 행렬 계산: $\Sigma = \frac{1}{n-1} X_{centered}^T X_{centered}$
3. 고유값 분해: $\Sigma = V \Lambda V^T$ (V=고유벡터, Λ=고유값)
4. 주성분 선택: 고유값이 큰 순서대로 k개 선택
5. 투영: $Z = X_{centered} V_k$

#### 설명 분산 비율 (Explained Variance Ratio)
$$EVR_k = \frac{\lambda_k}{\sum_{i=1}^{p} \lambda_i}$$

- $\lambda_k$: k번째 주성분의 고유값
- 보통 **누적 EVR ≥ 95%**가 되는 주성분 수 선택

In [ ]:
# ============================================================
# PCA 실습
# ============================================================
# 표준화 먼저 (PCA 전 필수!)
scaler_for_pca = StandardScaler()
X_scaled = scaler_for_pca.fit_transform(df_knn_imputed[selected_cols])

# PCA 적용
pca_full = PCA()
pca_full.fit(X_scaled)

print('=' * 60)
print('🔬 PCA 분석 결과')
print('=' * 60)

evr = pca_full.explained_variance_ratio_
cumulative_evr = np.cumsum(evr)

pca_info = pd.DataFrame({
    '주성분': [f'PC{i+1}' for i in range(len(evr))],
    '고유값': pca_full.explained_variance_.round(4),
    '설명 분산 비율 (%)': (evr * 100).round(2),
    '누적 설명 분산 비율 (%)': (cumulative_evr * 100).round(2)
})
print(pca_info.to_string(index=False))

# 최적 주성분 수 결정
n_components_95 = np.argmax(cumulative_evr >= 0.95) + 1
n_components_90 = np.argmax(cumulative_evr >= 0.90) + 1
print(f'\n90% 분산 보존 필요 주성분: {n_components_90}개')
print(f'95% 분산 보존 필요 주성분: {n_components_95}개')
print(f'(원본 특성 수: {len(selected_cols)}개)')

In [ ]:
# PCA 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Scree Plot (Elbow Method)
axes[0].bar(range(1, len(evr)+1), evr * 100, color='steelblue', alpha=0.8, label='개별 EVR')
axes[0].plot(range(1, len(cumulative_evr)+1), cumulative_evr * 100,
             'ro-', linewidth=2, markersize=6, label='누적 EVR')
axes[0].axhline(y=95, color='green', linestyle='--', alpha=0.7, label='95%')
axes[0].axhline(y=90, color='orange', linestyle='--', alpha=0.7, label='90%')
axes[0].set_xlabel('주성분 번호')
axes[0].set_ylabel('설명 분산 비율 (%)')
axes[0].set_title('Scree Plot\n(Elbow Method)', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].set_xticks(range(1, len(evr)+1))

# 2. 2D 시각화 (PC1 vs PC2)
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)
scatter = axes[1].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1],
                           c=y_target, cmap='bwr', alpha=0.6, s=20)
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title('PCA 2D 시각화\n(색상=타겟 클래스)', fontsize=11)
plt.colorbar(scatter, ax=axes[1], label='Target')

# 3. Loadings (주성분 계수) 히트맵
loadings = pd.DataFrame(
    pca_full.components_[:4].T,
    index=selected_cols,
    columns=[f'PC{i+1}' for i in range(4)]
)
sns.heatmap(loadings, ax=axes[2], annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 9})
axes[2].set_title('PCA Loadings\n(PC1~PC4)', fontsize=11)

plt.suptitle('PCA 분석 결과', fontsize=13)
plt.tight_layout()
plt.show()

# 차원 축소 적용
pca_optimal = PCA(n_components=n_components_95)
X_pca = pca_optimal.fit_transform(X_scaled)
print(f'\n✅ PCA 차원 축소 완료:')
print(f'  원본: {X_scaled.shape} → 축소: {X_pca.shape}')
print(f'  보존된 분산: {pca_optimal.explained_variance_ratio_.sum()*100:.1f}%')

---
## 3.2 Feature Importance (Random Forest)

### 📖 이론

**Feature Importance**는 각 특성이 모델 예측에 얼마나 기여하는지를 측정합니다.

#### Random Forest Feature Importance 계산

**불순도 감소 방법 (MDI, Mean Decrease in Impurity)**:

$$Importance(f) = \frac{1}{|T|} \sum_{t \in T} \sum_{n: split\_feature=f} p(n) \cdot \Delta I(n)$$

- $T$: 모든 트리의 집합
- $p(n)$: 노드 n에 도달하는 샘플 비율
- $\Delta I(n)$: 노드 n에서의 불순도 감소량

#### 불순도 지표
- **Gini Impurity (분류)**: $Gini = 1 - \sum_{k} p_k^2$
- **Entropy (분류)**: $H = -\sum_{k} p_k \log_2 p_k$
- **MSE (회귀)**: $MSE = \frac{1}{n} \sum (y_i - \bar{y})^2$

### 주의사항
- 카디널리티가 높은 특성이 과대평가될 수 있음
- 순열 중요도(Permutation Importance)가 더 신뢰성 높음
- SHAP 값으로 더 정밀한 분석 가능

In [ ]:
# ============================================================
# Feature Importance with Random Forest 실습
# ============================================================
from sklearn.model_selection import cross_val_score

X_clean = df_knn_imputed[selected_cols].values

# Random Forest 학습
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=5
)
rf.fit(X_clean, y_target)

# Feature Importance 추출
importances = rf.feature_importances_
std_importances = np.std([tree.feature_importances_ for tree in rf.estimators_], axis=0)

importance_df = pd.DataFrame({
    '특성': selected_cols,
    '중요도': importances,
    '표준편차': std_importances
}).sort_values('중요도', ascending=False)

print('=' * 60)
print('🌲 Random Forest Feature Importance')
print('=' * 60)
print(importance_df.to_string(index=False))
print(f'\n합계: {importances.sum():.4f} (합이 1이 되어야 함)')

# Cross-validation 점수
cv_scores = cross_val_score(rf, X_clean, y_target, cv=5)
print(f'\n5-fold CV 정확도: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# Feature Importance 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. 중요도 막대 그래프 (표준편차 포함)
sorted_idx = importance_df.index
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(selected_cols)))
bars = axes[0].barh(
    importance_df['특성'],
    importance_df['중요도'],
    xerr=importance_df['표준편차'],
    color=list(reversed(colors)),
    alpha=0.8,
    capsize=4
)
axes[0].set_xlabel('중요도 (Impurity Decrease)')
axes[0].set_title('Random Forest Feature Importance\n(error bar = 트리 간 표준편차)', fontsize=11)

# 중요도 값 표시
for i, (val, err) in enumerate(zip(importance_df['중요도'], importance_df['표준편차'])):
    axes[0].text(val + err + 0.002, i, f'{val:.3f}', va='center', fontsize=9)

# 2. 누적 중요도 (상위 k개로 충분한지 확인)
cumulative_imp = np.cumsum(importance_df['중요도'].values)
axes[1].plot(range(1, len(cumulative_imp)+1), cumulative_imp * 100,
             'bo-', linewidth=2, markersize=8)
axes[1].axhline(y=90, color='orange', linestyle='--', label='90%')
axes[1].axhline(y=95, color='green', linestyle='--', label='95%')
axes[1].fill_between(range(1, len(cumulative_imp)+1), cumulative_imp * 100,
                      alpha=0.2, color='steelblue')
axes[1].set_xlabel('상위 N개 특성')
axes[1].set_ylabel('누적 중요도 (%)')
axes[1].set_title('누적 Feature Importance\n(몇 개 특성으로 충분한가?)', fontsize=11)
axes[1].legend()
axes[1].set_xticks(range(1, len(cumulative_imp)+1))

plt.suptitle('Feature Importance 분석', fontsize=13)
plt.tight_layout()
plt.show()

# 상위 k개 특성 선택
top_k = np.argmax(cumulative_imp >= 0.95) + 1
top_features = importance_df['특성'].values[:top_k].tolist()
print(f'\n95% 중요도 커버를 위한 상위 {top_k}개 특성:')
print(top_features)

---
# 4️⃣ 실습 단계 (Practical) - 통합 Pipeline

## 4.1 sklearn Pipeline 개념

### 📖 이론

**Pipeline**은 여러 전처리 단계와 모델을 하나의 객체로 묶어 관리합니다.

```
원본 데이터
    ↓
[Step 1] KNN Imputer (결측치 처리)
    ↓
[Step 2] StandardScaler (표준화)
    ↓
[Step 3] PCA (차원 축소)
    ↓
[Step 4] RandomForest (분류)
    ↓
예측 결과
```

### Pipeline의 장점

| 장점 | 설명 |
|------|------|
| **데이터 누수 방지** | fit은 train에만, transform은 test에도 적용 |
| **코드 단순화** | 반복 코드 제거 |
| **재현성** | 동일한 전처리 보장 |
| **교차 검증** | Pipeline 전체에 CV 적용 가능 |

In [ ]:
# ============================================================
# 통합 Pipeline 구성
# ============================================================
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

# 데이터 분할
X_raw = df[selected_cols].values  # 결측치 포함 원본
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_target, test_size=0.2, random_state=42, stratify=y_target
)

print(f'학습 데이터: {X_train.shape}, 테스트 데이터: {X_test.shape}')

# Pipeline 구성
pipeline = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),       # Step 1: 결측치 처리
    ('scaler', StandardScaler()),                  # Step 2: 표준화
    ('pca', PCA(n_components=7)),                  # Step 3: 차원 축소
    ('classifier', RandomForestClassifier(        # Step 4: 분류
        n_estimators=100, random_state=42, max_depth=5
    ))
])

# 학습 및 평가
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print('\n' + '=' * 60)
print('📊 Pipeline 모델 성능')
print('=' * 60)
print(classification_report(y_test, y_pred, target_names=['Class 0', 'Class 1']))

# 교차 검증
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X_raw, y_target, cv=cv, scoring='accuracy')
print(f'5-fold CV 정확도: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'개별 점수: {[round(s, 4) for s in cv_scores]}')

In [ ]:
# ============================================================
# 단계별 처리 비교 (전처리 효과 확인)
# ============================================================
results = {}

# 1. 원본 데이터 (결측치 평균 대치만)
pipe_baseline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])
scores_baseline = cross_val_score(pipe_baseline, X_raw, y_target, cv=5)
results['베이스라인\n(평균 대치만)'] = scores_baseline

# 2. KNN 대치 + 표준화
pipe_knn_std = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])
scores_knn_std = cross_val_score(pipe_knn_std, X_raw, y_target, cv=5)
results['KNN 대치\n+ 표준화'] = scores_knn_std

# 3. KNN 대치 + 표준화 + PCA
pipe_full = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=7)),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])
scores_full = cross_val_score(pipe_full, X_raw, y_target, cv=5)
results['KNN 대치\n+ 표준화\n+ PCA'] = scores_full

# 4. 전체 Pipeline
pipe_complete = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('selector', SelectKBest(f_classif, k=15)),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])
scores_complete = cross_val_score(pipe_complete, X_raw, y_target, cv=5)
results['완전 Pipeline\n(+Poly+Select)'] = scores_complete

print('=' * 70)
print('📊 단계별 Feature Engineering 효과 비교')
print('=' * 70)
for name, scores in results.items():
    clean_name = name.replace('\n', ' ')
    print(f'{clean_name:35s}: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# 최종 결과 비교 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 1. 박스 플롯 비교
labels = [name.replace('\n', '\n') for name in results.keys()]
data_plot = list(results.values())
bp = axes[0].boxplot(data_plot, labels=labels, patch_artist=True,
                      medianprops={'color': 'red', 'linewidth': 2})
colors_box = ['lightblue', 'lightgreen', 'lightyellow', 'lightsalmon']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[0].set_ylabel('정확도 (Accuracy)')
axes[0].set_title('Pipeline 단계별 성능 비교\n(5-fold CV)', fontsize=11)
axes[0].tick_params(axis='x', labelsize=8)

# 2. 평균 ± 표준편차 비교
means = [s.mean() for s in results.values()]
stds = [s.std() for s in results.values()]
x_pos = range(len(results))
axes[1].bar(x_pos, means, yerr=stds, capsize=8,
            color=colors_box, edgecolor='black', alpha=0.8)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(labels, fontsize=8)
axes[1].set_ylabel('평균 정확도')
axes[1].set_title('평균 정확도 ± 표준편차', fontsize=11)
for i, (m, s) in enumerate(zip(means, stds)):
    axes[1].text(i, m + s + 0.005, f'{m:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# 기준선
for ax in axes:
    ax.axhline(y=means[0], color='gray', linestyle=':', alpha=0.7, label='베이스라인')

plt.suptitle('Feature Engineering 전처리 단계별 효과\n(클수록 좋음)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# 개선도 계산
print('\n' + '=' * 60)
print('📈 베이스라인 대비 개선도')
print('=' * 60)
baseline_mean = means[0]
for name, mean in zip(results.keys(), means):
    clean_name = name.replace('\n', ' ')
    improvement = mean - baseline_mean
    arrow = '↑' if improvement > 0.001 else ('↓' if improvement < -0.001 else '→')
    print(f'{clean_name:35s}: {mean:.4f} ({arrow} {improvement:+.4f})')

In [ ]:
# 혼동 행렬 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 최종 Pipeline 성능
pipeline.fit(X_train, y_train)
y_pred_final = pipeline.predict(X_test)
y_prob_final = pipeline.predict_proba(X_test)[:, 1]

# 혼동 행렬
cm = confusion_matrix(y_test, y_pred_final)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])
axes[0].set_title('혼동 행렬 (Confusion Matrix)', fontsize=11)

# ROC 곡선
from sklearn.metrics import roc_curve, auc
fpr, tpr, _ = roc_curve(y_test, y_prob_final)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC 곡선', fontsize=11)
axes[1].legend()

plt.suptitle('최종 Pipeline 성능 평가', fontsize=13)
plt.tight_layout()
plt.show()

---
# 📚 정리 및 요약

## Feature Engineering 선택 가이드

```
데이터 수신
    ↓
결측치 확인 ──────────────────────────────────────────┐
    │ 결측치 있음                                      │
    ├─ 5% 미만 → 제거                                   │
    ├─ 5~30% → 평균/중앙값/최빈값 대치                  │
    └─ 30% 이상 → KNN/모델 기반 대치 또는 피처 제거     │
         ↓                                             │
이상치 확인                                            │
    ├─ 이상치 많음 → Robust Scaling                    │
    └─ 이상치 적음 → Standard/MinMax Scaling            │
         ↓                                             │
스케일 차이                                            │
    ├─ [0,1] 범위 필요 → Min-Max                       │
    ├─ 정규분포 가정 → Standardization                  │
    └─ 이상치 강건 필요 → Robust Scaling                │
         ↓                                             │
범주형 변수                                            │
    ├─ 순서 없음 & 카디널리티 낮음 → One-Hot            │
    └─ 트리 모델 or 순서 있음 → Label Encoding          │
         ↓                                             │
고차원 데이터                                          │
    ├─ 분산 보존 우선 → PCA                            │
    └─ 중요 특성 선택 → SelectKBest/Feature Importance  │
         ↓                                             │
비선형 관계 필요 → Polynomial Features ←────────────────┘
```

## 스케일링 방법 요약

| 방법 | 수식 | 범위 | 이상치 | 추천 모델 |
|------|------|------|--------|----------|
| Min-Max | $(x-x_{min})/(x_{max}-x_{min})$ | [0,1] | 민감 | 신경망, KNN |
| Standardization | $(x-\mu)/\sigma$ | (-∞,+∞) | 보통 | 선형모델, PCA |
| Robust | $(x-Q_2)/(Q_3-Q_1)$ | (-∞,+∞) | 강건 | 이상치 많은 경우 |

## 핵심 원칙

1. **항상 train 데이터로만 fit**, test 데이터는 transform만
2. **Pipeline 사용** → 데이터 누수 방지
3. **도메인 지식 활용** → 의미 있는 Feature 생성
4. **반복적 실험** → 여러 방법 비교 후 선택
5. **시각화 필수** → 변환 전후 분포 확인

---
**🎓 Feature Engineering 완전 완성! 수고하셨습니다!** 🎉